Building A Chatbot 

How to design and implement an LLM-powered chatbot.This chatbot will be able to have a conversation and remember previous interactions.

Note: This chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for: 
1. Conversational RAG: Enable a chatbot experience over an external source of data
2. Agents: Build a chatbot that can take actions


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")
 

In [2]:
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-20b",groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.0', 'langchain': '1.5.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000015328E0B770>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000015328F90A10>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage,SystemMessage
model.invoke([HumanMessage(content="Hi, My name is Ayush and I persuing my B.tech from NIT Patna in ECE department")])

AIMessage(content='Hello Ayush! 👋  \nIt’s great to meet you. How can I help you today? Whether you’re looking for study tips, project ideas, career guidance, or anything else related to your B.Tech in ECE, feel free to let me know!', additional_kwargs={'reasoning_content': 'User says: "Hi, My name is Ayush and I persuing my B.tech from NIT Patna in ECE department". They are greeting. Likely they want some help. We should respond politely, ask how we can help.'}, response_metadata={'token_usage': {'completion_tokens': 114, 'prompt_tokens': 94, 'total_tokens': 208, 'completion_time': 0.140018014, 'completion_tokens_details': {'reasoning_tokens': 51}, 'prompt_time': 0.037423881, 'prompt_tokens_details': None, 'queue_time': 0.632057099, 'total_time': 0.177441895}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d23c14756c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1e2c8-ab69-726a-8ca4-1b198ebd3996-0', to

In [4]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi, My name is Ayush and I am a B.tech Student"),
        AIMessage(content="Hello Ayush! 👋  \nNice to meet you. How can I help you today? Whether it’s a question about your coursework, a project you’re working on, or anything else, just let me know!"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content='You’re Ayush, a B.Tech student. Let me know what you’d like to chat about or need help with!', additional_kwargs={'reasoning_content': 'User says: "Hey What\'s my name and what do I do?" They earlier said their name is Ayush, B.Tech student. So answer: name Ayush, B.Tech student. Should respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 149, 'total_tokens': 226, 'completion_time': 0.085120838, 'completion_tokens_details': {'reasoning_tokens': 43}, 'prompt_time': 0.007220556, 'prompt_tokens_details': None, 'queue_time': 0.283432945, 'total_time': 0.092341394}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_565badff47', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1e2c8-bae4-73cf-bd45-0683d559b4e3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 149, 'output_tokens': 77, 'total_tokens': 226, 'output_token_details': 

## Message History
We can use a message History class to wrap our model and make it stateful.This will keep track of inputs and outputs of the model, and store them in some datadtore.Future interactions will then load those messages and pass them into the chain as part of the input.

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id ]   

with_message_history=RunnableWithMessageHistory(model,get_session_history)


C:\Users\ayush gupta\AppData\Local\Temp\ipykernel_25504\2748200497.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
c:\Users\ayush gupta\OneDrive\Desktop\LANGCHAIN\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
config={"configurable":{"session_id":"chat1"}}

In [7]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Ayush and I am a B.tech Student")],
    config=config 
)

C:\Users\ayush gupta\AppData\Local\Temp\ipykernel_25504\2748200497.py:9: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id]=ChatMessageHistory()


In [8]:
response.content

'Hi Ayush! 👋 How can I help you today?'

In [9]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config 
)

AIMessage(content='Your name is Ayush.', additional_kwargs={'reasoning_content': 'User: "What\'s my name?" They previously said "Hi, My name is Ayush and I am a B.tech Student". So answer: Ayush.'}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 112, 'total_tokens': 160, 'completion_time': 0.062847973, 'completion_tokens_details': {'reasoning_tokens': 33}, 'prompt_time': 0.006424341, 'prompt_tokens_details': None, 'queue_time': 0.156615988, 'total_time': 0.069272314}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a12402de73', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1e2c8-fd64-7490-b610-4ef65ff70ea4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 112, 'output_tokens': 48, 'total_tokens': 160, 'output_token_details': {'reasoning': 33}})

In [10]:
# change the config--> session_id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config1
)

response.content

'I’m not sure—I don’t have that information. Could you let me know your name?'

In [11]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey my name is Ayush")],
    config=config1
)
response.content

'Nice to meet you, Ayush! How can I help you today?'

In [12]:
response=with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config1
)

response.content

'Your name is Ayush.'

# Prompt templates

Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case,the raw user input is just a message,which  we are passing to the LLM.

In [13]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer all the question to the best of your ability in {language}."),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [14]:
response=chain.invoke({"messages":[HumanMessage(content="Hi my name is Ayush")],"language":"Hindi"})
response.content

'नमस्ते आयुष! आपका स्वागत है। मैं आपकी किस प्रकार सहायता कर सकता हूँ?'

In [ ]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history,input_messages_key="messages")


c:\Users\ayush gupta\OneDrive\Desktop\LANGCHAIN\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [16]:
config ={"configurable":{"session_id":"chat4"}}
response=with_message_history.invoke(
    {"messages":[HumanMessage(content="Hi, I am Ayush")],"language":"Hindi"},
    config=config
)

response.content

'नमस्ते आयुष! आपका स्वागत है। मैं आपकी कैसे मदद कर सकता हूँ?'

In [17]:
response=with_message_history.invoke(
    {"messages":[HumanMessage(content="whats my name?")],"language":"Hindi"},
    config=config,
)

response.content

'आपका नाम आयुष है।'

# Managing the Conversation History



In [23]:
from transformers import AutoTokenizer
from langchain_core.messages import trim_messages

tokenizer = AutoTokenizer.from_pretrained("gpt2")

def count_tokens(messages):
    return sum(
        len(tokenizer.encode(str(message.content)))
        for message in messages
    )

trimmer = trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter=count_tokens,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

trimmed_messages = trimmer.invoke(messages)

for message in trimmed_messages:
    print(message)

c:\Users\ayush gupta\OneDrive\Desktop\LANGCHAIN\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ayush gupta\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


content="You're a good assistant" additional_kwargs={} response_metadata={}
content="Hi! I'm bob" additional_kwargs={} response_metadata={}
content='hi!' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
content='I like vanilla ice cream' additional_kwargs={} response_metadata={}
content='nice' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
content='whats 2+2' additional_kwargs={} response_metadata={}
content='4' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
content='thanks' additional_kwargs={} response_metadata={}
content='no problem!' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
content='having fun?' additional_kwargs={} response_metadata={}
content='yes!' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]


In [28]:

from operator import itemgetter

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant. Answer in {language}."
    ),
    MessagesPlaceholder(variable_name="messages")
])

chain = (
    RunnablePassthrough.assign(
        messages=itemgetter("messages") | trimmer
    )
    | prompt
    | model
)

response = chain.invoke({
    "messages": messages + [
        HumanMessage(content="What ice cream do I like?")
    ],
    "language": "English"
})

print(response.content)


You like vanilla ice cream.


In [29]:
## Lats wrap this on the messages History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)

config={"configurable":{"session_id":"chat5"}}

c:\Users\ayush gupta\OneDrive\Desktop\LANGCHAIN\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [30]:
response = with_message_history.invoke({
    "messages": messages + [HumanMessage(content="whats my name?")],
    "language":"English",
},
config=config 
)

response.content

'Your name is Bob.'